# Suite CHG — Changes: what a transaction changed, observed at its commit, as a journal

A journal, a viewer or a layout engine needs to know exactly what a transaction changed, which its expression, a query that writes, does not say ([CAP-270: a transaction's changes observed at its commit](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-270)). `Changes.observe(store, observer)` gives `observer` a journal of each transaction that commits having changed something, a `Changes.Journal` of that one transaction, and returns a function that ends it. The journal is written in the journal language, the `Changes` schemas every store holds ([the journals proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/JOURNALS.md#the-changes-schemas-proposed)): a transaction's `id`, the uid `Provenance` records it by; its symbols, each bound to an object held before, by its reference, or to a new object, by its schema's; then its changes, in order, relation entries removed and inserted and properties set and cleared, net of what the transaction undid within itself, a counted object written by its value where an entry links it. A journal is data, written, read and compared as any value is. `Changes.apply(txn, journal)` applies it: to the store as it was, it makes the same changes; applied again, none.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Changes, Comparison, JSON as Json, Plain, Provenance, Proxies, References } from "@mbse/schemas/Framework";
import { DecodeError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

/** crm.Person, owned by the singleton book by key, with a name and a home, who knows persons since a year, and pins
 * persons through no adjacency by name; crm.Note, counted, about a person. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Home: S.OfObject, Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject,
      Note: S.OfObject, About: S.OfRelation, Pinned: S.OfRelation })
    .spec([
      (_: any) => _.Home.name("crm.Home").properties({ city: S.OfStr, zip: S.OfStr }),
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")])
        .properties({ since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, home: _.Home }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
        (t: any) => t.relation([_.Pinned]).me("pinner"), // no adjacency to crm.Pinned has a name
        (t: any) => t.relation([_.Pinned]).me("pinned"),
      ]),
      (_: any) => _.Pinned.name("crm.Pinned").links([(t: any) => t.name("pinner"), (t: any) => t.name("pinned")]),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")])
        .properties({ key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }).adjacencies({
        about: (t: any) => t.relation([_.About]).me("note") }),
      (_: any) => _.About.name("crm.About").links([(t: any) => t.name("note"), (t: any) => t.name("person")]),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(crm);

/** The contact the book holds by `key`, within the transaction open: found again by navigating from the singleton. */
function contact(key: string): any {
  return store.singleton("crm.TheBook").contacts({ key });
}

/** A path, short: its singleton, then each key it passes, `crm.TheBook/ada`; a schema's, its name. */
function short(path: any): string {
  const parts: string[] = [path.get("singleton")];
  for (let at = path.get("next") ?? new Map(); at.has("entry"); at = at.get("entry").get("next") ?? new Map()) {
    parts.push(...[...(at.get("entry").get("properties") ?? new Map()).values()].map(String));
  }
  return parts[0] === "." ? parts.slice(1).join(".") : parts.join("/");
}

/** A journal of `source`'s, the crm store unless given, as plain data. */
function plain(journal: unknown, source: any = store): any {
  return Plain.ToPlain(source)(source.registered("Changes.Journal"), journal);
}

/** A schema object's path, as a `References.OfSchema` writes it: from the root module through each module and the
 * schema, by name. */
function schemaPath(...names: string[]): Record<string, unknown> {
  let nested: Record<string, unknown> = { target: {} };
  for (const name of [...names].reverse()) nested = { entry: { i: "schemas", o: "", properties: { name }, next: nested } };
  return { path: { singleton: ".", next: nested } };
}

/** A journal of one transaction of `source`, the crm store unless given, short: each symbol, as what it is bound to,
 * then each change, in order; an object an entry change names by its symbol, by its reference, short, or, a counted
 * object, inline, by its values. */
function shown(journal: unknown, source: any = store): string[] {
  const [transaction] = plain(journal, source);
  const linked = (link: any): string => {
    const [[kind, given]] = [...link];
    if (kind === "counted") {
      const values = [...(given.get("properties") ?? new Map())].map(([name, value]: any) => `${name}=${Json.dumps(value)}`);
      return `(${short(given.get("schema").get("path"))} ${values.join(" ")})`;
    }
    return kind === "reference" ? `@${short(given.get("path"))}` : given;
  };
  const lines: string[] = [...transaction.get("symbols")].map(([name, bound]: any) => bound.has("reference")
    ? `${name} = ${short(bound.get("reference").get("path"))}` : `${name} = new ${short(bound.get("schema").get("path"))}`);
  for (const change of transaction.get("changes")) {
    const [[kind, action]] = [...change];
    const [[verb, body]] = [...action];
    if (kind === "property") {
      lines.push(`${body.get("holder")}.${body.get("name")} = ` + (verb === "set" ? Json.dumps(body.get("value")) : "cleared"));
      continue;
    }
    const parts = [...[...(body.get("links") ?? new Map())].map(([n, link]: any) => `${n}=${linked(link)}`),
      ...[...(body.get("properties") ?? new Map())].map(([n, v]: any) => `${n}=${Json.dumps(v)}`)];
    lines.push(`${verb === "insert" ? "+" : "-"} ${linked(body.get("holder"))}.${body.get("adjacency")}(${parts.join(", ")})`);
  }
  return lines;
}

/** A transaction of `source`'s, the crm store unless given, adding persons to the book, each keyed by its name. */
function people(names: string[], source: any = store): any {
  const [book, person] = [source.registered("crm.Book"), source.registered("crm.Person")];
  return (t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ book, ...Object.fromEntries(names.map((n) => [n, person])) })
    .spec([(_: any) => _.book.contacts(Object.fromEntries(names.map((n) => [n, (e: any) => e.contact(_[n])]))),
      ...names.map((n) => (_: any) => _[n].name(n))])]))]);
}

/** Applies in `target` a journal of `source`'s, the crm store unless given, carried as its plain form. */
function applied(target: any, journal: unknown, source: any = store): void {
  const carried = Json.loads(Json.dumps(plain(journal, source)));
  const read = Plain.FromPlain(target)(target.registered("Changes.Journal"), carried);
  target.transaction((txn: any) => Changes.apply(txn, read));
}

/** A store's data, its schemas aside, as JSON text. */
const data = (s: any): string => Json.dumps(Plain.ToPlain(s).Snapshot()[1] as any);
const log: any[] = [];
const end = Changes.observe(store, (journal) => log.push(journal));

<a id="CHG-01"></a>
## CHG-01 · Each transaction that commits having changed something is given to the observer as a journal of that one transaction, after its commit, a value that outlives it; one that changes nothing is not

In [ ]:
store.transaction(people(["ada", "bob"]));
assert(equal(shown(log.at(-1)), ["o0 = crm.TheBook", "o1 = new crm.Person", "o2 = new crm.Person",
  '+ o0.contacts(contact=o1, key="ada")', '+ o0.contacts(contact=o2, key="bob")', 'o1.name = "ada"', 'o2.name = "bob"']));
assert(log.at(-1).as_schema() === store.registered("Changes.Journal") && log.at(-1).length === 1); // schema data, read outside
{
  const count = log.length;
  store.transaction(people(["ada"])); // changes nothing: the spec holds
  assert(log.length === count);
}

<a id="CHG-02"></a>
## CHG-02 · A transaction that raises, its writes dropped, is observed by none

In [ ]:
{
  const count = log.length;
  raises(Error, () => store.transaction(() => {
    store.crm.Person(contact("ada"), (p: any) => p.name("Zed"));
    throw new Error("refused");
  }), "refused");
  store.transaction(() => assert(log.length === count && contact("ada").name.get() === "ada"));
}

<a id="CHG-03"></a>
## CHG-03 · What a transaction undid within itself is no change: an entry inserted and removed, a value set and set back

In [ ]:
{
  const count = log.length;
  store.transaction(() => {
    const [ada, bob] = [contact("ada"), contact("bob")];
    store.crm.Person(ada, (p: any) => p.knows((a: any) => a.insert((e: any) => e.known(bob).since(2001n))));
    store.crm.Person(ada, (p: any) => p.knows((a: any) => a.remove((e: any) => e.known(bob))));
    store.crm.Person(ada, (p: any) => p.name("Ada"));
    store.crm.Person(ada, (p: any) => p.name("ada"));
  });
  assert(log.length === count);
}

<a id="CHG-04"></a>
## CHG-04 · Removing an owning entry deletes what it owned: the journal removes that entry by its owner's key, naming nothing deleted, and what goes with it follows

In [ ]:
store.transaction(() => {
  store.crm.Person(contact("ada"), (p: any) => p.knows((a: any) => a.insert((e: any) => e.known(contact("bob")).since(2001n))));
});
assert(equal(shown(log.at(-1)), ["o0 = crm.TheBook/bob", "o1 = crm.TheBook/ada", "+ o1.knows(known=o0, since=2001)"]));
store.transaction(() => {
  store.crm.Book(store.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("bob"))));
});
assert(equal(shown(log.at(-1)), ["o0 = crm.TheBook", '- o0.contacts(key="bob")']));

<a id="CHG-05"></a>
## CHG-05 · A property changed is set to its value, written whole and bare by its schema, a value object's part included, or cleared

In [ ]:
store.transaction(() => {
  store.crm.Person(contact("ada"), (p: any) => p.home((h: any) => h.city("Paris").zip("75001")));
});
store.transaction(() => {
  store.crm.Person(contact("ada"), (p: any) => p.home((h: any) => h.city("Lyon")));
});
assert(equal(shown(log.at(-1)), ["o0 = crm.TheBook/ada", 'o0.home = {"city": "Lyon", "zip": "75001"}']));
store.transaction(() => {
  store.crm.Person(contact("ada"), (p: any) => p.name(null));
});
assert(equal(shown(log.at(-1)), ["o0 = crm.TheBook/ada", "o0.name = cleared"]));

<a id="CHG-06"></a>
## CHG-06 · A counted object, immutable, is written by its value, inline, in the entry change that links it, never bound to a symbol: found by its value where the journal is applied, or made

In [ ]:
store.transaction(() => {
  store.crm.Note((n: any) => n.text("hello").about((a: any) => a.insert((e: any) => e.person(contact("ada")))));
});
assert(equal(shown(log.at(-1)), ["o0 = crm.TheBook/ada", '+ (crm.Note text="hello").about(person=o0)']));
{
  const [change] = plain(log.at(-1))[0].get("changes");
  assert(equal(change.get("entry").get("insert").get("holder"), map({ counted: { schema: schemaPath("crm", "Note"),
    properties: { text: "hello" } } }))); // its schema and its values
}

<a id="CHG-07"></a>
## CHG-07 · Each journal, written and read as plain data and applied in order to a store as the first was (`Changes.apply`), makes it the same, applied twice the same again; once the observing ends, none is given

In [ ]:
const replica: any = new B.Store();
replica.transaction(crm);
for (const journal of log) for (let i = 0; i < 2; i += 1) applied(replica, journal);
assert(data(replica) === data(store));
end();
end(); // ending it again changes nothing
{
  const count = log.length;
  store.transaction(people(["eve"]));
  assert(log.length === count);
}

<a id="CHG-08"></a>
## CHG-08 · An entry that no object it links names an adjacency to cannot be written in the journal: while observed, its transaction is refused, naming its relation

In [ ]:
{
  const pinned = store.registered("crm.Pinned");
  const stop = Changes.observe(store, () => undefined);
  raises(ValueError, () => store.transaction(() => { // only the framework's own writes reach such a relation
    const ada = contact("ada");
    Proxies._add_entry(pinned, Proxies._make_entry(pinned, new Map([["pinner", ada], ["pinned", ada]]), new Map()));
  }), "no object an entry of 'crm.Pinned' links has an adjacency to it by name");
  stop();
  store.transaction(() => assert(Proxies._linking(pinned, "pinner", contact("ada")).length === 0));
}

<a id="CHG-09"></a>
## CHG-09 · Objects of many schemas owned through one relation: deleting one is written by its owner's key alone, with what it owned and every entry linking it following, applied twice the same; an owner with no adjacency by name to the relation cannot write it

In [ ]:
/** org.Team and org.Member, each owned through org.Owns, by name, as is org.Badge, a member's; members are buddies;
 * org.Key is owned by the singleton vault through no adjacency by name. */
function org(txn: any): any {
  const S = txn.Schemas;
  const owned = (t: any, _: any): any => t.name("owner").relation([_.Owns]).me("owned").owner(true);
  return txn.uid("org/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Owns: S.OfRelation, Root: S.OfObject, Team: S.OfObject, Member: S.OfObject, Badge: S.OfObject,
      Buddy: S.OfRelation, Holds: S.OfRelation, Vault: S.OfObject, Key: S.OfObject })
    .spec([
      (_: any) => _.Owns.name("org.Owns").links([(t: any) => t.name("owner"), (t: any) => t.name("owned")])
        .properties({ name: S.OfStr }).uniques([["owner", "name"], ["owned"]]),
      (_: any) => _.Buddy.name("org.Buddy").links([(t: any) => t.name("a"), (t: any) => t.name("b")]),
      (_: any) => _.Holds.name("org.Holds").links([(t: any) => t.name("vault"), (t: any) => t.name("key")])
        .properties({ name: S.OfStr }).uniques([["vault", "name"], ["key"]]),
      (_: any) => _.Root.name("org.Root").ref(true).singleton("org.TheRoot").adjacencies({
        owns: (t: any) => t.relation([_.Owns]).me("owner") }),
      (_: any) => _.Team.name("org.Team").ref(true).adjacencies([
        (t: any) => owned(t, _), (t: any) => t.name("owns").relation([_.Owns]).me("owner")]),
      (_: any) => _.Member.name("org.Member").ref(true).adjacencies([
        (t: any) => owned(t, _), (t: any) => t.name("owns").relation([_.Owns]).me("owner"),
        (t: any) => t.name("buddies").relation([_.Buddy]).me("a"),
        (t: any) => t.relation([_.Buddy]).me("b")]),
      (_: any) => _.Badge.name("org.Badge").ref(true).properties({ code: S.OfStr }).adjacencies([
        (t: any) => owned(t, _)]),
      (_: any) => _.Vault.name("org.Vault").ref(true).singleton("org.TheVault").adjacencies([
        (t: any) => t.relation([_.Holds]).me("vault")]), // no adjacency by name
      (_: any) => _.Key.name("org.Key").ref(true).adjacencies([
        (t: any) => t.name("vault").relation([_.Holds]).me("key").owner(true)]),
    ])]))]);
}

const first: any = new B.Store();
const second: any = new B.Store();
for (const s of [first, second]) s.transaction(org);
const journals: any[] = [];
Changes.observe(first, (journal) => journals.push(journal));
first.transaction(() => {
  const root = first.singleton("org.TheRoot");
  const team = first.org.Team((t: any) => t.owner((a: any) => a.insert((e: any) => e.name("core").owner(root))));
  const ann = first.org.Member((m: any) => m.owner((a: any) => a.insert((e: any) => e.name("ann").owner(team))));
  first.org.Member((m: any) => m.owner((a: any) => a.insert((e: any) => e.name("ben").owner(team)))
    .buddies((a: any) => a.insert((e: any) => e.b(ann))));
  first.org.Badge((b: any) => b.owner((a: any) => a.insert((e: any) => e.name("pass").owner(ann))).code("A1"));
});
const team = (): any => first.singleton("org.TheRoot").owns({ name: "core" }); // found again within each transaction
const member = (name: string): any => team().owns({ name });
first.transaction(() => {
  first.org.Member(member("ben"), (m: any) => m.buddies((a: any) => a.remove((e: any) => e.b(member("ann")))));
});
assert(equal(shown(journals.at(-1), first), ["o0 = org.TheRoot/core/ann", "o1 = org.TheRoot/core/ben", "- o1.buddies(b=o0)"]));
first.transaction(() => {
  first.org.Member(member("ben"), (m: any) => m.buddies((a: any) => a.insert((e: any) => e.b(member("ann")))));
});
first.transaction(() => {
  first.org.Team(team(), (t: any) => t.owns((a: any) => a.remove((e: any) => e.name("ann"))));
});
assert(equal(shown(journals.at(-1), first), ["o0 = org.TheRoot/core", '- o0.owns(name="ann")']));
for (const journal of journals) for (let k = 0; k < 2; k++) applied(second, journal, first);
assert(data(second) === data(first));
first.transaction(() => {
  assert(equal([...team().owns].map((e: any) => e.name.get()), ["ben"]) && [...member("ben").buddies].length === 0);
});
first.transaction(() => {
  first.org.Key((k: any) => k.vault((a: any) => a.insert((e: any) => e.name("k1").vault(first.singleton("org.TheVault")))));
});
raises(ValueError, () => first.transaction(() => {
  const [held] = Proxies._linking(first.registered("org.Holds"), "vault", first.singleton("org.TheVault")); // no adjacency by name
  const key = held.key;
  first.org.Key(key, (k: any) => k.vault((a: any) => a.clear()));
}), "no owner of an entry of 'org.Holds' has an adjacency to it by name");

<a id="CHG-10"></a>
## CHG-10 · The journal language's schemas are every store's own, as the reference schemas are, value schemas the store was made holding; installed again, nothing changes. Each kind of change is a union of its actions: an entry change inserts or removes, a property change sets the whole value or clears it; an entry change names each object as a `Changes.OfLink`, a symbol, an object's reference, a counted object's value or a parameter reference, and a schema by its `References.OfSchema`

In [ ]:
{
  const fresh: any = new B.Store();
  const names = ["Changes.Journal", "Changes.Transaction", "Changes.Symbol", "Changes.Change", "Changes.OfEntry",
    "Changes.OfEntry.Insert", "Changes.OfEntry.Remove", "Changes.OfProperty", "Changes.OfProperty.Set",
    "Changes.OfProperty.Clear", "Changes.OfLink", "Changes.Counted"];
  assert(names.every((name) => fresh.framework(fresh.registered(name)) && !fresh.registered(name).ref));
  assert(fresh.registered("Changes.Journal") !== store.registered("Changes.Journal")); // each store's own
  const before = names.map((name) => fresh.registered(name));
  fresh.transaction(Changes.install); // found by their names
  assert(names.every((name, i) => fresh.registered(name) === before[i]));
  assert(equal([...fresh.registered("Changes.Transaction").properties.keys()], ["id", "symbols", "changes"]));
  const branches = (name: string): string[] => fresh.registered(name).branches.map((branch: any) => branch.name);
  assert(equal(branches("Changes.Change"), ["entry", "property"]) && equal(branches("Changes.OfEntry"), ["insert", "remove"]));
  assert(equal(branches("Changes.OfProperty"), ["set", "clear"]));
  assert(equal(branches("Changes.OfLink"), ["symbol", "reference", "counted", "parameter"]));
  assert(["Insert", "Remove"].every((action) => equal([...fresh.registered(`Changes.OfEntry.${action}`).properties.keys()],
    ["holder", "adjacency", "links", "properties"])));
  assert(equal([...fresh.registered("Changes.OfProperty.Set").properties.keys()], ["holder", "name", "value"]));
  assert(equal([...fresh.registered("Changes.OfProperty.Clear").properties.keys()], ["holder", "name"]));
  assert(equal([...fresh.registered("Changes.Counted").properties.keys()], ["schema", "properties"]));
  const OfSchema = fresh.registered("References.OfSchema");
  assert(fresh.registered("Changes.Symbol").properties.get("schema").schema === OfSchema);
  assert(fresh.registered("Changes.Counted").properties.get("schema").schema === OfSchema);
}

<a id="CHG-11"></a>
## CHG-11 · A transaction's journal has an id, unique: the uid `Provenance` records the transaction by, its expression's where it gives one, else one the store makes, whether or not `Provenance` is installed

In [ ]:
{
  const ids: string[] = [];
  const stop = Changes.observe(store, (journal) => ids.push(plain(journal)[0].get("id")));
  store.transaction((t: any) => people(["fay"])(t.uid("add fay")));
  store.transaction(() => {
    store.crm.Person(contact("fay"), (p: any) => p.name("Fay"));
  });
  store.transaction(() => {
    store.crm.Person(contact("fay"), (p: any) => p.name("fay"));
  });
  stop();
  assert(ids[0] === "add fay" && new Set(ids).size === 3 && ids.every((i) => typeof i === "string" && i.length > 0));
  const recorded: any = new B.Store();
  recorded.transaction(crm);
  recorded.transaction(Provenance.install);
  recorded.transaction((t: any) => Provenance.extend(t, [recorded.registered("crm.Person")]));
  const seen: string[] = [];
  Changes.observe(recorded, (journal) => seen.push(plain(journal, recorded)[0].get("id")));
  recorded.transaction(() => {
    const book = recorded.singleton("crm.TheBook");
    recorded.crm.Book(book, (b: any) => b.contacts({ gus: (e: any) => e.contact((p: any) => p.name("gus")) }));
  });
  recorded.transaction(() => { // the transaction recorded by the id its journal has
    assert([...recorded.singleton("Provenance.Transactions").transactions].map((e: any) => e.uid.get()).includes(seen.at(-1)));
  });
}

<a id="CHG-12"></a>
## CHG-12 · A journal is data, with no store of its own: its transactions read by iterating it, written and read as plain data the same, and compared as any value is; whole transactions taken from it, in order, are a journal, which applies alone, and applies again changing nothing; a journal whose later transaction deletes what an earlier one made applies at once; a journal that says what no journal can is refused

In [ ]:
{
  const [source, target]: any[] = [new B.Store(), new B.Store()];
  for (const s of [source, target]) s.transaction(crm);
  const taken: any[] = [];
  const stop = Changes.observe(source, (journal) => taken.push(journal));
  source.transaction(people(["ann", "cal"], source));
  source.transaction(() => {
    const book = source.singleton("crm.TheBook");
    source.crm.Person(book.contacts({ key: "ann" }), (p: any) => p.knows((a: any) => a.insert(
      (e: any) => e.known(book.contacts({ key: "cal" })).since(1999n))));
  });
  source.transaction(() => {
    source.crm.Person(source.singleton("crm.TheBook").contacts({ key: "cal" }), (p: any) => p.name("Cal"));
  });
  stop();
  const whole = taken.flatMap((journal) => plain(journal, source)); // one journal of all three
  const Journal = target.registered("Changes.Journal");
  const journal = Plain.FromPlain(target)(Journal, Json.loads(Json.dumps(whole)));
  const read: unknown[] = [];
  journal.as_entries((items: any) => {
    for (const [, t] of items) t.as_object((o: any) => read.push(o.id.get()));
  });
  assert(equal(read, whole.map((t: any) => t.get("id"))) && journal.length === 3); // iterated, in order
  assert(Json.dumps(Plain.ToPlain(target)(Journal, journal) as any) === Json.dumps(whole)); // written back the same
  const again = Plain.FromPlain(target)(Journal, whole);
  assert(new Comparison.OfContainer(Journal, journal).compare(new Comparison.OfContainer(Journal, again)) === 0);
  const part = Plain.FromPlain(target)(Journal, whole.slice(0, 2)); // its first two transactions: a journal
  assert(new Comparison.OfContainer(Journal, journal).compare(new Comparison.OfContainer(Journal, part)) === null);
  for (let i = 0; i < 2; i += 1) target.transaction((txn: any) => Changes.apply(txn, part));
  target.transaction(() => {
    const held = target.singleton("crm.TheBook");
    assert(equal([...held.contacts].map((e: any) => [e.key.get(), e.contact.name.get()]), [["ann", "ann"], ["cal", "cal"]]));
    assert(equal([...held.contacts({ key: "ann" }).knows].map((e: any) => [e.known.name.get(), e.since.get()]), [["cal", 1999n]]));
  });
  target.transaction((txn: any) => Changes.apply(txn, Plain.FromPlain(target)(Journal, whole.slice(2))));
  assert(data(target) === data(source));
  target.transaction((txn: any) => Changes.apply(txn, Plain.FromPlain(target)(Journal, []))); // nothing to apply
  assert(data(target) === data(source));
  const path = (key: string): any => ({ singleton: "crm.TheBook", next: { entry: {
    i: "contacts", o: "book", properties: { key }, next: { target: {} } } } });
  const symbols = { c0: { reference: { path: path("ann") } } }; // a symbol named as a counted object would be: the
  const note = { counted: { schema: schemaPath("crm", "Note"), properties: { text: "met" } } }; // counted object's is
  const noted = map([{ id: "n1" }, { id: "n2", symbols: { ...symbols, o1: { reference: { path: path("cal") } } }, changes: [
    { entry: { insert: { holder: note, adjacency: "about", links: { person: { symbol: "c0" } } } } }, // another
    { entry: { insert: { holder: note, adjacency: "about", links: { person: { symbol: "o1" } } } } }] }]);
  for (let i = 0; i < 2; i += 1) target.transaction((txn: any) => Changes.apply(txn, Plain.FromPlain(target)(Journal, noted)));
  target.transaction(() => {
    const ann = target.singleton("crm.TheBook").contacts({ key: "ann" });
    const [about] = Proxies._linking(target.registered("crm.About"), "person", ann) as any[];
    const [also] = Proxies._linking(target.registered("crm.About"), "person",
      target.singleton("crm.TheBook").contacts({ key: "cal" })) as any[];
    assert(about.note.text.get() === "met" && also.note === about.note); // one note, of one value, found again
  });
  const wrong = Json.loads(Json.dumps(noted)) as any;
  wrong[1].get("changes")[0].get("entry").get("insert").set("holder", map({ parameter: { name: "p" } })); // no object's
  raises(ValueError, () => target.transaction((txn: any) => Changes.apply(txn, Plain.FromPlain(target)(Journal, wrong))),
    "an entry change is written on an object's adjacency: its holder names an object"); // adjacency writes it
  const empty = target.Changes.Journal((j: any) => j.at(0n).as_object((t: any) => t.id("e").changes([(c: any) => c.entry(
    (e: any) => e.insert((i: any) => i.holder((link: any) => link).adjacency("about")))]))); // its holder names nothing
  raises(ValueError, () => target.transaction((txn: any) => Changes.apply(txn, empty)),
    "a union of the journal language holds one of its branches, and one here holds none");
  raises(DecodeError, () => target._arrive(map({ id: "n3" })), // a journal, as plain data, is a list of its transactions
    "a dense container must be a list, got dict");
  const [made, gone]: any[] = [new B.Store(), new B.Store()];
  for (const s of [made, gone]) s.transaction(crm);
  const kept: any[] = [];
  const stopKept = Changes.observe(made, (journal) => kept.push(journal));
  made.transaction(people(["bob"], made));
  made.transaction(() => {
    made.crm.Book(made.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("bob"))));
  });
  stopKept();
  const both = kept.flatMap((journal) => plain(journal, made)); // made, then deleted
  gone.transaction((txn: any) => Changes.apply(txn, Plain.FromPlain(gone)(gone.registered("Changes.Journal"), both)));
  gone.transaction(() => { // applied at once: the person made and deleted, its frame binding refused
    assert([...gone.extent("crm.Person")].length === 0);
  });
}

<a id="CHG-13"></a>
## CHG-13 · A value in a change is written whole by the schema of the property it is the value of, bare, a value of a schema with no name of its own among them, and marked with its schema only where that schema cannot tell it, a refinement's; a counted object's values by its schema's; read back the same; one of a property the holder's schema lacks is a value of any schema, tagged, and refused bare

In [ ]:
{
  /** val.Thing, the singleton val.TheThing, with tags, a list of text, a point or a refinement of one, a number or a
   * text, and a count; val.Mark, counted, marks a thing by a weight. */
  const val = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("val/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Point: S.OfObject, Point3: S.OfObject, Thing: S.OfObject, Mark: S.OfObject, Marks: S.OfRelation })
      .spec([
        (_: any) => _.Point.name("val.Point").properties({ x: S.OfInt, y: S.OfInt }),
        (_: any) => _.Point3.name("val.Point3").refines([_.Point]).properties({ z: S.OfInt }),
        (_: any) => _.Marks.name("val.Marks").links([(t: any) => t.name("mark"), (t: any) => t.name("thing")])
          .properties({ weight: S.OfInt }),
        (_: any) => _.Thing.name("val.Thing").ref(true).singleton("val.TheThing").properties({
          tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])),
          at: _.Point,
          either: S.OfUnion((u: any) => u.branches([(t: any) => t.name("n").schema([S.OfInt]),
            (t: any) => t.name("s").schema([S.OfStr])])),
          count: S.OfInt,
        }).adjacencies({ marks: (t: any) => t.relation([_.Marks]).me("thing") }),
        (_: any) => _.Mark.name("val.Mark").ref(true).properties({ label: S.OfStr }).adjacencies({
          on: (t: any) => t.relation([_.Marks]).me("mark") }),
      ])]))]);
  };
  const values: any = new B.Store();
  values.transaction(val);
  const thing = { reference: { path: { singleton: "val.TheThing", next: { target: {} } } } };
  const verbs = map([{ id: "v1", symbols: { x: thing }, changes: [
    { property: { set: { holder: "x", name: "tags", value: ["a", "b"] } } },
    { property: { set: { holder: "x", name: "at", value: { $schema: "val.Point3", x: 1n, y: 2n, z: 3n } } } },
    { property: { set: { holder: "x", name: "either", value: { s: "one" } } } },
    { property: { clear: { holder: "x", name: "count" } } },
    { entry: { insert: { holder: { symbol: "x" }, adjacency: "marks", links: { mark: { counted: {
      schema: schemaPath("val", "Mark"), properties: { label: "m" } } } }, properties: { weight: 7n } } } },
    { entry: { insert: { holder: { counted: { schema: schemaPath("val", "Mark"), properties: { label: "n" } } },
      adjacency: "on", links: { thing: { symbol: "x" } }, properties: { weight: 8n } } } },
  ] }]);
  const Values = values.registered("Changes.Journal");
  assert(equal(Plain.ToPlain(values)(Values, Plain.FromPlain(values)(Values, verbs)), verbs)); // bare by its schema, the same
  assert(Json.dumps(Plain.ToPlain(values)(Values, Plain.FromPlain(values)(Values, verbs)) as any) === Json.dumps(verbs));
  const unknown = map([{ id: "v2", symbols: { x: thing }, changes: [
    { property: { set: { holder: "x", name: "size", value: { int: 9n } } } },
    { entry: { insert: { holder: { symbol: "x" }, adjacency: "marks", properties: { height: { int: 2n } } } } }] }]);
  assert(equal(Plain.ToPlain(values)(Values, Plain.FromPlain(values)(Values, unknown)), unknown)); // tagged: no schema says
  unknown[0].get("changes")[0].get("property").get("set").set("value", 9n);
  raises(DecodeError, () => Plain.FromPlain(values)(Values, unknown), "a value of any schema is {schema: value}");
}

<a id="CHG-14"></a>
## CHG-14 · `store.follow(follower)` gives each transaction the store commits as a journal of that one transaction, as plain data, the journal `Changes.observe` gives; a cache layer over it applies it to what it holds, under its id (suite BLW), a counted object by its value

In [ ]:
{
  const below: any = new B.Store();
  below.transaction(crm);
  const above: any = new B.Store(below); // a cache layer over it, following it
  above.transaction(crm);
  const [given, followed]: any[][] = [[], []];
  Changes.observe(below, (journal) => given.push(journal));
  const stop = below.follow((journal) => followed.push(journal));
  below.transaction(people(["hal"], below)); // made by another: made above too, under the book it holds
  stop();
  assert(equal(followed, [plain(given[0], below)]) && followed[0][0].get("changes")[0].get("entry").get("insert").get("adjacency") === "contacts");
  above.transaction(() => above.singleton("crm.TheBook").contacts({ key: "hal" })); // present above
  const aboveGiven: any[] = [];
  Changes.observe(above, (journal) => aboveGiven.push(journal));
  below.transaction(() => {
    below.crm.Person(below.singleton("crm.TheBook").contacts({ key: "hal" }), (p: any) => p.name("Hal"));
  });
  above.transaction(() => assert(above.singleton("crm.TheBook").contacts({ key: "hal" }).name.get() === "Hal"));
  const [arrived] = aboveGiven;
  assert(equal(shown(arrived, above), ["o0 = crm.TheBook/hal", 'o0.name = "Hal"']));
  assert(plain(arrived, above)[0].get("id") === plain(given.at(-1), below)[0].get("id")); // under the id it arrived with
  const unnamed = plain(given.at(-1), below).map((t: any) => new Map([...t].filter(([k]: any) => k !== "id")));
  above._arrive(Json.loads(Json.dumps(unnamed))); // a journal whose transaction has no id, as a follow gives none: held,
  assert(aboveGiven.length === 1); // it changes nothing
  below.transaction(() => { // a counted object, about one held above: made above, by its value
    below.crm.Note((n: any) => n.text("hi").about((a: any) => a.insert((e: any) => e.person(
      below.singleton("crm.TheBook").contacts({ key: "hal" })))));
  });
  above.transaction(() => {
    const [about] = Proxies._linking(above.registered("crm.About"), "person",
      above.singleton("crm.TheBook").contacts({ key: "hal" })) as any[];
    assert(about.note.text.get() === "hi" && aboveGiven.length === 2);
  });
}

<a id="CHG-15"></a>
## CHG-15 · An entry change names its holder and its links by an object's reference as well as by a symbol of its transaction, a counted object by its value: a journal so written inserts and removes the entries it names, applied twice the same; built through the builders, it is the same value


In [ ]:
{
  const linked: any = new B.Store();
  linked.transaction(crm);
  linked.transaction(people(["ida", "jon"], linked));
  const Linked = linked.registered("Changes.Journal");
  const at = (key: string): any => ({ path: { singleton: "crm.TheBook", next: { entry: {
    i: "contacts", o: "book", properties: { key }, next: { target: {} } } } } });
  const byReference = map([{ id: "r1", changes: [
    { entry: { insert: { holder: { reference: at("ida") }, adjacency: "knows",
      links: { known: { reference: at("jon") } }, properties: { since: 2010n } } } },
    { entry: { insert: { holder: { counted: { schema: schemaPath("crm", "Note"), properties: { text: "seen" } } },
      adjacency: "about", links: { person: { reference: at("jon") } } } } }] },
  { id: "r2", changes: [{ entry: { remove: { holder: { reference: at("ida") }, adjacency: "knows",
    links: { known: { reference: at("jon") } } } } }] }]) as any[];
  const read = Plain.FromPlain(linked)(Linked, byReference);
  assert(equal(Plain.ToPlain(linked)(Linked, read), byReference)); // a payload read by the entry's schema, named by reference
  const bare: any = new B.Store();
  bare.transaction(crm); // holding none of the objects it names: read by the schema each path implies
  const Bare = bare.registered("Changes.Journal");
  assert(equal(Plain.ToPlain(bare)(Bare, Plain.FromPlain(bare)(Bare, byReference)), byReference));
  for (let i = 0; i < 2; i += 1) {
    linked.transaction((txn: any) => Changes.apply(txn, Plain.FromPlain(linked)(Linked, byReference.slice(0, 1))));
  }
  const [ida, jon] = linked.transaction(() => {
    const book = linked.singleton("crm.TheBook");
    assert(equal([...book.contacts({ key: "ida" }).knows].map((e: any) => [String(e.known.name), e.since.get()]),
      [["jon", 2010n]]));
    const [about] = Proxies._linking(linked.registered("crm.About"), "person", book.contacts({ key: "jon" })) as any[];
    assert(about.note.text.get() === "seen"); // one note, found again by its value
    return ["ida", "jon"].map((key) => References.of(linked, book.contacts({ key })));
  });
  const built = linked.Changes.Journal((j: any) => j.at(0n).as_object((t: any) => t.id("r2").changes([(c: any) => c.entry(
    (e: any) => e.remove((r: any) => r.holder((h: any) => h.reference(ida)).adjacency("knows").links({
      known: (k: any) => k.reference(jon) })))])));
  assert(equal(Plain.ToPlain(linked)(Linked, built), byReference.slice(1))); // through the builders, the same value
  for (let i = 0; i < 2; i += 1) linked.transaction((txn: any) => Changes.apply(txn, built));
  linked.transaction(() => assert([...linked.singleton("crm.TheBook").contacts({ key: "ida" }).knows].length === 0));
}


<a id="CHG-16"></a>
## CHG-16 · A property whose schema is a container with no name of its own: its change, its whole value, is written bare by that schema as the transaction that writes it commits, applies in another store, and a cache layer writes it through to the layer below


In [ ]:
{
  /** lst.Shelf, the singleton lst.TheShelf, with tags, a list of text, a container with no name of its own. */
  const lists = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("lst/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Shelf: S.OfObject }).spec([
        (_: any) => _.Shelf.name("lst.Shelf").ref(true).singleton("lst.TheShelf").properties({
          tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])) }),
      ])]))]);
  };
  const [shelf, copy]: any[] = [new B.Store(), new B.Store()];
  for (const s of [shelf, copy]) s.transaction(lists);
  const kept: any[] = [];
  Changes.observe(shelf, (journal) => kept.push(journal));
  shelf.transaction(() => shelf.lst.Shelf(shelf.singleton("lst.TheShelf"), (s: any) => s.tags(["a", "b"])));
  const [transaction] = plain(kept.at(-1), shelf);
  assert(equal(transaction.get("changes"), map([{ property: { set: { holder: "o0", name: "tags", value: ["a", "b"] } } }])));
  for (let i = 0; i < 2; i += 1) applied(copy, kept.at(-1), shelf);
  assert(data(copy) === data(shelf));
  const under: any = new B.Store();
  under.transaction(lists);
  const over: any = new B.Store(under); // a cache layer over it
  over.transaction(lists);
  over.transaction(() => { // written through, as it is made
    over.lst.Shelf(over.singleton("lst.TheShelf"), (s: any) => s.tags(["c"]));
  });
  assert(data(under) === data(over) && data(under).includes('"c"'));
}
